# VLM QLoRA Training — Kaggle Session (Resume)

## Before each session

1. **Update your `vlm-session-state` dataset** with the latest checkpoint from your
   last run (the `lora_stepXXXX/` folder + `cls_head.pt` + `faiss_index/` + `stage2.jsonl`).
2. **Attach BOTH datasets** to this notebook (right panel → Add Input):
   - `vlm-projector`     → `projector_stage1.pt`
   - `vlm-session-state` → latest checkpoint + faiss_index + cls_head + log
3. **Add Kaggle Secrets** (Add-ons → Secrets):
   - `HF_TOKEN`
   - `SEMANTIC_SCHOLAR_API_KEY`
4. **Settings → Accelerator → GPU T4 ×2** (only GPU 0 is used; that's fine).
5. **Run All.**

## Notes
- Cell 5 auto-discovers files anywhere under `/kaggle/input` — mount path can't break it.
- The 4 GB VRAM hypothesis is already confirmed, so this runs at full 16 GB speed.
- Cell 8 auto-saves to `/kaggle/working/session_state/` when training stops (timeout or finish).
- After the session: download `session_state/` from the Output tab → update the
  `vlm-session-state` dataset for next time.
- `IS_FIRST_SESSION = False` for every resume (it already is, below).


In [ ]:
# ── USER CONFIG — edit these before each session ─────────────────────────────

IS_FIRST_SESSION  = False  # True = session 1 (builds FAISS, no prior checkpoint)
                            # False = session 2+ (restores FAISS + checkpoint)

# 4GB hypothesis already confirmed — always use full 16 GB from now on
SIMULATE_4GB_VRAM = False

# Kaggle Dataset slugs — must match what you created in your Kaggle account
PROJECTOR_DATASET = "vlm-projector"       # contains projector_stage1.pt
STATE_DATASET     = "vlm-session-state"   # contains checkpoint + faiss_index

# Your Kaggle username (needed to build input paths)
KAGGLE_USERNAME   = "sujalprasad"

# Training config
MAX_PAIRS  = 4000
EPOCHS     = 3
GRAD_ACCUM = 4      # full speed (hypothesis confirmed, no longer need 8)
SAVE_EVERY = 250
LR         = 2e-4

print("Config loaded.")
print(f"  Session type   : {'FIRST (builds FAISS)' if IS_FIRST_SESSION else 'RESUME'}")
print(f"  VRAM mode      : Full 16 GB (4GB hypothesis confirmed)")
print(f"  Grad accum     : {GRAD_ACCUM}")
print(f"  Kaggle user    : {KAGGLE_USERNAME}")


In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
# Kaggle already has torch, transformers, datasets, numpy, Pillow
# We only need the extras

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("peft>=0.19.1")
pip("bitsandbytes>=0.49.2")
pip("accelerate>=1.13.0")
pip("faiss-cpu==1.13.2")
pip("sentence-transformers")
pip("python-dotenv")

print("Packages ready.")


In [ ]:
# ── Secrets & environment variables ──────────────────────────────────────────

import os, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"]                 = secrets.get_secret("HF_TOKEN")
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = secrets.get_secret("SEMANTIC_SCHOLAR_API_KEY")
os.environ["CUBLAS_WORKSPACE_CONFIG"]  = ":4096:8"

# HuggingFace login (needed for MIMIC credentialed access)
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

# ── 4 GB cap (must be set BEFORE any CUDA allocation) ─────────────────────────
if SIMULATE_4GB_VRAM:
    # T4 = 16 GB → fraction 0.25 = 4 GB hard ceiling.
    # We use 3.8/16 not 4/16 to account for ~200 MB Windows WDDM overhead
    # that exists on the real laptop but not on Kaggle Linux — keeps the
    # simulation honest.
    torch.cuda.set_per_process_memory_fraction(3.8 / 16, 0)
    os.environ["MEDDIAG_MAX_VRAM_GB"] = "3.8"
    print("4 GB VRAM simulation ACTIVE")
    print(f"  Fraction set : {3.8/16:.4f}  ({3.8:.1f} GB / 16 GB T4)")
    print(f"  Any allocation beyond 3.8 GB will OOM exactly like an RTX 3050")
else:
    os.environ["MEDDIAG_MAX_VRAM_GB"] = "14"
    print("Full 16 GB mode — no VRAM cap")

print("Secrets loaded, HF login OK.")


In [ ]:
# ── Clone repo & set working directory ───────────────────────────────────────

import subprocess, os

REPO_URL  = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_DIR  = "/kaggle/working/vlm"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
    print(f"Repo cloned → {REPO_DIR}")
else:
    # Discard local patches to run_pipeline.sh before pulling (cell 7 re-applies them)
    subprocess.call(["git", "-C", REPO_DIR, "checkout", "run_pipeline.sh"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print(f"Repo updated → {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ── Copy assets from Kaggle input datasets into working tree ─────────────────
# Kaggle's mount path varies (/kaggle/input/<slug>/ or
# /kaggle/input/datasets/<user>/<slug>/), so we AUTO-DISCOVER the files by
# walking /kaggle/input instead of hardcoding a path. This is the part that
# broke in past sessions — searching makes it bulletproof.

import shutil, re, os
from pathlib import Path

MODELS_DIR = Path(REPO_DIR) / "models"
MODELS_DIR.mkdir(exist_ok=True)

def _find_file(filename: str) -> Path | None:
    """Return first match for `filename` anywhere under /kaggle/input."""
    for root, _dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            return Path(root) / filename
    return None

def _find_dir(dirname: str) -> Path | None:
    """Return first directory named `dirname` anywhere under /kaggle/input."""
    for root, dirs, _files in os.walk("/kaggle/input"):
        if dirname in dirs:
            return Path(root) / dirname
    return None

# ── Projector (always needed) ─────────────────────────────────────────────────
proj_src = _find_file("projector_stage1.pt")
if proj_src is None:
    raise FileNotFoundError(
        "projector_stage1.pt not found under /kaggle/input — "
        "did you attach the 'vlm-projector' dataset?"
    )
shutil.copy2(proj_src, MODELS_DIR / "projector_stage1.pt")
print(f"Projector copied  ({proj_src.stat().st_size / 1e6:.0f} MB)  from {proj_src.parent}")

if not IS_FIRST_SESSION:
    # ── FAISS index ────────────────────────────────────────────────────────────
    faiss_src = _find_dir("faiss_index")
    if faiss_src is not None:
        faiss_dst = Path(REPO_DIR) / "faiss_index"
        if faiss_dst.exists():
            shutil.rmtree(faiss_dst)
        shutil.copytree(faiss_src, faiss_dst)
        print(f"FAISS index restored → {faiss_dst}  (from {faiss_src})")
    else:
        print("WARNING: faiss_index not found — Stage 2 may rebuild it.")

    # ── LoRA checkpoint(s) ──────────────────────────────────────────────────────
    ckpt_pattern = re.compile(r"lora_step(\d+)$")
    found_ckpts = []
    for root, dirs, _files in os.walk("/kaggle/input"):
        for d in dirs:
            if ckpt_pattern.match(d):
                found_ckpts.append(Path(root) / d)
    if not found_ckpts:
        raise FileNotFoundError(
            "No lora_stepXXXX checkpoint found under /kaggle/input — "
            "did you attach the 'vlm-session-state' dataset?"
        )
    for src in found_ckpts:
        dst = MODELS_DIR / src.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Checkpoint restored  → {dst.name}")

    # ── ClassificationHead ──────────────────────────────────────────────────────
    cls_src = _find_file("cls_head.pt")
    if cls_src is not None:
        shutil.copy2(cls_src, MODELS_DIR / "cls_head.pt")
        print("ClassificationHead   restored")
    else:
        print("WARNING: cls_head.pt not found — will init fresh (loses cls progress).")

    # ── Training log ────────────────────────────────────────────────────────────
    logs_dir = Path(REPO_DIR) / "logs"
    logs_dir.mkdir(exist_ok=True)
    log_src = _find_file("stage2.jsonl")
    if log_src is not None:
        shutil.copy2(log_src, logs_dir / "stage2.jsonl")
        print("Training log         restored")

print("\nAssets ready.")


In [ ]:
# ── Fix pipeline state for Kaggle ─────────────────────────────────────────────
# The repo's .pipeline_state already has steps 0,1,2 marked done.
# On session 1 we need step 1 (FAISS build) to run since we have no index yet.

from pathlib import Path

state_path = Path(REPO_DIR) / "logs" / ".pipeline_state"
state_path.parent.mkdir(exist_ok=True)

if IS_FIRST_SESSION:
    # Write state with only step0 and step2 done; step1 (FAISS) must rebuild
    state_path.write_text("\nstep0\nstep0\nstep0\nstep0\nstep0\nstep0\nstep0\nstep2\n")
    print("Pipeline state: FAISS will be rebuilt (session 1)")
else:
    # Keep the repo's state as-is (step1 already done — FAISS was restored above)
    print(f"Pipeline state: using repo defaults (FAISS + stage 1 already done)")

# Remove stale lock if any
lock = Path(REPO_DIR) / "logs" / ".pipeline.lock"
lock.unlink(missing_ok=True)
print("Lock cleared.")


In [ ]:
# ── Patch run_pipeline.sh with Kaggle-optimised config ───────────────────────
# Overwrites GRAD_ACCUM and MAX_PAIRS inline; no permanent file change needed.

pipeline_sh = Path(REPO_DIR) / "run_pipeline.sh"
text = pipeline_sh.read_text()

import re
text = re.sub(r"(GRAD_ACCUM=)\d+",   f"GRAD_ACCUM={GRAD_ACCUM}",   text)
text = re.sub(r"(MAX_PAIRS_S2=)\d+", f"MAX_PAIRS_S2={MAX_PAIRS}",  text)

pipeline_sh.write_text(text)
print(f"run_pipeline.sh patched: GRAD_ACCUM={GRAD_ACCUM}, MAX_PAIRS_S2={MAX_PAIRS}")


In [ ]:
# ── Run training, then AUTO-PACKAGE + AUTO-UPLOAD to your Kaggle dataset ───────
# After training stops (finish OR you interrupt the cell), this:
#   1. copies the 2 latest checkpoints + cls_head + faiss + log into session_state/
#   2. pushes session_state/ as a new version of your vlm-session-state dataset
# => NO manual download. Next session's cell 5 pulls it automatically.
#
# NOTE: a HARD 12h Kaggle kill stops the kernel before this runs. To be safe,
# interrupt this cell yourself before the limit (the upload then runs), or just
# let training finish.

import subprocess, os, shutil, re, json
from pathlib import Path

env = os.environ.copy()
cmd = ["bash", "run_pipeline.sh", "--resume"]

print("Starting pipeline — full 16 GB mode...")
print("=" * 60)

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\n[notebook] Interrupted — packaging + uploading checkpoint now...")
proc.wait()
print(f"\n[notebook] Process exited with code {proc.returncode}")


def _kaggle_auth() -> bool:
    """Set up Kaggle CLI auth from the KAGGLE_KEY secret.
    Supports BOTH formats:
      - new access token  : value starts with 'KGAT_'  -> KAGGLE_API_TOKEN
      - classic API key   : 32-char hex                -> KAGGLE_USERNAME + KAGGLE_KEY
    Returns True if a credential was set.
    """
    try:
        token = secrets.get_secret("KAGGLE_KEY").strip()
    except Exception as e:
        print(f"\n  !! KAGGLE_KEY secret missing ({e}). Skipping upload.")
        return False

    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    if token.startswith("KGAT_"):
        # New token-based auth
        os.environ["KAGGLE_API_TOKEN"] = token
        kdir = Path.home() / ".kaggle"
        kdir.mkdir(exist_ok=True)
        at = kdir / "access_token"
        at.write_text(token)
        at.chmod(0o600)
        print("  Kaggle auth: using access token (KGAT_).")
    else:
        # Classic username+key auth
        os.environ["KAGGLE_KEY"] = token
        kdir = Path.home() / ".kaggle"
        kdir.mkdir(exist_ok=True)
        kj = kdir / "kaggle.json"
        kj.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": token}))
        kj.chmod(0o600)
        print("  Kaggle auth: using classic API key.")
    return True


def package_and_upload():
    # ── 1. Package latest state into session_state/ ───────────────────────────
    OUT_DIR = Path("/kaggle/working/session_state")
    OUT_DIR.mkdir(exist_ok=True)
    MODELS  = Path(REPO_DIR) / "models"
    LOGS    = Path(REPO_DIR) / "logs"

    ckpt_pattern = re.compile(r"lora_step(\d+)$")
    ckpts = sorted([(int(m.group(1)), p) for p in MODELS.iterdir()
                    if (m := ckpt_pattern.match(p.name))])
    for _, ckpt_path in ckpts[-2:]:
        dst = OUT_DIR / ckpt_path.name
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(ckpt_path, dst)
        print(f"  packaged {ckpt_path.name}")

    cls = MODELS / "cls_head.pt"
    if cls.exists(): shutil.copy2(cls, OUT_DIR / "cls_head.pt"); print("  packaged cls_head.pt")

    faiss_src = Path(REPO_DIR) / "faiss_index"
    if faiss_src.exists():
        fdst = OUT_DIR / "faiss_index"
        if fdst.exists(): shutil.rmtree(fdst)
        shutil.copytree(faiss_src, fdst); print("  packaged faiss_index/")

    log = LOGS / "stage2.jsonl"
    if log.exists(): shutil.copy2(log, OUT_DIR / "stage2.jsonl"); print("  packaged stage2.jsonl")

    # ── 2. Upload as a new version of the Kaggle dataset ──────────────────────
    if not _kaggle_auth():
        print(f"     Files are still in {OUT_DIR} — download from Output tab as fallback.")
        return

    meta = {"title": STATE_DATASET,
            "id": f"{KAGGLE_USERNAME}/{STATE_DATASET}",
            "licenses": [{"name": "CC0-1.0"}]}
    (OUT_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    r = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", "auto-update from training session", "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    print(r.stdout); print(r.stderr)
    if r.returncode == 0:
        print("\n✓ Dataset updated automatically — NO manual download needed.")
    else:
        print("\n✗ Upload failed. Files are in", OUT_DIR, "— download from Output tab.")


print("\n[notebook] Packaging + uploading session state...")
package_and_upload()


In [ ]:
# ── MANUAL force-upload (optional) ────────────────────────────────────────────
# Run this ANY time to push the current latest checkpoint to your Kaggle dataset
# WITHOUT stopping training. Useful as a mid-session safety snapshot.
# (Cell 8 already does this automatically when training stops.)

package_and_upload()


## No more manual downloads

Cell 8 auto-uploads your checkpoint to the `vlm-session-state` dataset when
training stops. Next session just **Run All** — cell 5 pulls the latest version.

### One-time setup for the auto-upload
1. https://www.kaggle.com/settings → **API** → **Create New Token** → `kaggle.json` downloads.
2. Open it, copy the `key` value.
3. **Add-ons → Secrets** → add `KAGGLE_KEY` = that value.

If `KAGGLE_KEY` is missing, cell 8 skips the upload and leaves the files in
`/kaggle/working/session_state/` so you can still grab them from the Output tab.

### Mid-session safety snapshot
Run **cell 9** any time to force-push the current checkpoint without stopping
training.

### Hard 12h limit
A hard kill stops the kernel before cell 8's upload runs. So either let training
finish, or **interrupt cell 8 yourself** before the limit — the upload then runs.

## Speed reference (full 16 GB T4)
| Phase | Speed |
|---|---|
| Warmup (~first 100 steps) | 1–4 s/step (rising, ignore) |
| Steady state | ~8–10 s/step |
| Per 12h session | ~4,300–5,400 steps |
